# CLIMP retraining (CS613) — Kaggle runner
Retrains CLIMP from the paper's starting points (VMamba-B ImageNet-1K + frozen Mamba-1.4B) on COCO Captions, then
evaluates it with the same scripts as the released checkpoints. All logic lives in `src/train_climp.py`.
Settings: **GPU T4 x2, Internet On**, secrets **`HF_TOKEN`** and **`HF_WRITE_TOKEN`** (write access; data and
checkpoints are handed between runs through a private HF repo `<you>/climp-cs613-train`).

One step per run, in order — check each step's zip before starting the next:
1. `STEP = "check"` (~45 min): weight checks, speed/batch probe, and a tiny end-to-end run (prep → train → resume → eval).
2. `STEP = "prep"` (~45 min): COCO train2017 → 256 px images + cached caption features → HF repo.
3. `STEP = "train"` (hours): both GPUs train one model; checkpoint + COCO-val progress every 45 min. If it stops on the time budget, rerun with `RESUME = True`.
4. `STEP = "eval"` with `EVAL = ["E1E2"]`, then `["E3"]`, then `["E5"]` (two entries run in parallel, one per GPU).

Download `/kaggle/working/train_<STEP>.zip` from the Output tab after every run.

In [ ]:
!nvidia-smi
!df -h /tmp /kaggle/working | cat
!free -g

In [ ]:
import os, subprocess, sys, threading, time
from pathlib import Path

STEP = "check"     # "check" -> "prep" -> "train" -> "eval"   (one step per Kaggle run)
EPOCHS = 3         # train: set from the check step's speed probe
BATCH = 64         # train: per GPU (global batch = 2x); set from the probe
GRAD_CKPT = False  # train: True if BATCH only fits with gradient checkpointing
MAX_HOURS = 10.5   # train: stop + checkpoint before Kaggle's 12 h limit
RESUME = False     # train: continue from the last checkpoint in the HF repo
EVAL = ["E1E2"]    # eval: any of "E1E2", "E3", "E5"; up to two, run in parallel (one per GPU)

REPO_URL = "https://github.com/tanishwanve-git/climp-cs613.git"
UPSTREAM_COMMIT = "3ff1279bb87f905596e6187f0e4a14f2cd28e818"
REPO = Path("/kaggle/working/climp-cs613")
LOGS = Path("/kaggle/working/logs")
os.environ["DATA_DIR"] = "/tmp/data"
os.environ["HF_HOME"] = "/tmp/data/hf"
os.environ["NCCL_P2P_DISABLE"] = "1"  # safer NCCL on cloud T4 pairs; negligible cost here
from kaggle_secrets import UserSecretsClient
for name in ["HF_TOKEN", "HF_WRITE_TOKEN"]:
    os.environ[name] = UserSecretsClient().get_secret(name)
print("tokens loaded | STEP =", STEP)

In [ ]:
t0 = time.time()
!rm -rf {REPO} && git clone -q {REPO_URL} {REPO}
!git clone -q https://github.com/NimrodShabtay/CLIMP.git {REPO}/CLIMP && git -C {REPO}/CLIMP checkout -q {UPSTREAM_COMMIT}
!echo "repo: $(git -C {REPO} rev-parse --short HEAD)  upstream: $(git -C {REPO}/CLIMP rev-parse --short HEAD)"

# 1) torch 2.10 (CUDA 12.6): the newest torch with official prebuilt mamba-ssm / causal-conv1d wheels.
#    Kaggle's newer torch has none, which forces a 40-60 min compile. torchaudio must match torch, because
#    transformers imports it if it is installed.
!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu126
# 2) prebuilt CUDA kernels for torch 2.10 + CUDA 12 + this Python
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
WHEELS = [
    f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1+cu12torch2.10cxx11abiTRUE-{PY}-{PY}-linux_x86_64.whl",
    f"https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-{PY}-{PY}-linux_x86_64.whl",
]
!pip install -q {" ".join(WHEELS)}
# 3) everything else: upstream requirements (minus torch / kernels) + ours
!grep -viE "^(torch|torchvision|triton|causal-conv1d|mamba-ssm)" {REPO}/CLIMP/requirements.txt > /tmp/upstream_reqs.txt
!pip install -q -r /tmp/upstream_reqs.txt -r {REPO}/requirements.txt

# 4) self-test: the kernels must actually run on the T4
Path("/tmp/check_kernels.py").write_text('''
import torch, transformers, mamba_ssm, causal_conv1d
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
from causal_conv1d import causal_conv1d_fn
import selective_scan_cuda
d = "cuda"
y = selective_scan_fn(torch.randn(1, 4, 8, device=d), torch.rand(1, 4, 8, device=d), -torch.rand(4, 1, device=d),
                      torch.randn(1, 1, 8, device=d), torch.randn(1, 1, 8, device=d))
z = causal_conv1d_fn(torch.randn(1, 8, 16, device=d), torch.randn(8, 4, device=d))
torch.cuda.synchronize()
print("kernels OK on", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| mamba_ssm", mamba_ssm.__version__, "| causal_conv1d", causal_conv1d.__version__)
''')
!python /tmp/check_kernels.py
if _exit_code != 0:
    print("!!! prebuilt kernels failed -> compiling from source (slow, 40-60 min)")
    !pip install -q causal-conv1d mamba-ssm --no-build-isolation
    !python /tmp/check_kernels.py
# 5) every library the experiment scripts import must load; stop here (not 20 min later) if one does not
Path("/tmp/check_imports.py").write_text('''
import torch, torchvision, transformers, open_clip, timm, webdataset, fvcore, scipy, matplotlib, pyarrow
from transformers import AutoTokenizer, MambaForCausalLM, Mamba2ForCausalLM
import sys; sys.path.insert(0, "CLIMP")
from models import load_climp, build_vmamba, MambaCLIP
print("imports OK | torchvision", torchvision.__version__, "| transformers", transformers.__version__, "| open_clip", open_clip.__version__)
''')
!cd {REPO} && python /tmp/check_imports.py
if _exit_code != 0:  # transformers imports torchaudio whenever it is installed; this project never uses it
    print("!!! import check failed -> uninstalling torchaudio and retrying")
    !pip uninstall -y -q torchaudio
    !cd {REPO} && python /tmp/check_imports.py
assert _exit_code == 0, "an import failed: fix the environment before running experiments"
print(f"install total: {time.time() - t0:.0f}s")

In [ ]:
import psutil

OUT = "results_trained_smoke" if STEP == "check" else "results_trained"
T = "python src/train_climp.py"
TRAIN = "torchrun --nproc_per_node 2 src/train_climp.py train"
EVALS = {"E1E2": "python src/exp_nocaps.py --models climp-ours --out-dir results_trained && "
                 "python src/exp_geometry.py --models climp-ours --out-dir results_trained",
         "E3": "python src/exp_zeroshot.py --models climp-ours --out-dir results_trained",
         "E5": "python src/exp_docci.py --models climp-ours --tokens full --out-dir results_trained"}

def final_weights(tag):
    """Our trained weights (vision tower + projections) from the HF artifact repo."""
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=os.environ["HF_WRITE_TOKEN"])
    return hf_hub_download(f"{api.whoami()['name']}/climp-cs613-train", f"{tag}/final/model.safetensors",
                           token=api.token, local_dir="/tmp/data/ours")

# each stage: list of parallel jobs (name, command, GPUs visible); stages run one after another
prep = lambda s: [[("prep_images", f"{T} prep-images{s}", ""),
                   ("prep_text0", f"{T} prep-text{s} --part 0 --parts 2", "0"),
                   ("prep_text1", f"{T} prep-text{s} --part 1 --parts 2", "1")],
                  [("prep_upload", f"{T} prep-upload{s}", "")]]
if STEP == "check":
    STAGES = [[("check", f"{T} check", "0")]] + prep(" --smoke") + [
        [("train_smoke", f"{TRAIN} --smoke --epochs 3 --batch-size 32 --max-hours 0.04 --check-every-min 2", "0,1")],
        [("train_smoke_resume", f"{TRAIN} --smoke --epochs 3 --batch-size 32 --resume --check-every-min 2", "0,1")],
        [("eval_smoke", "CLIMP_OURS_CKPT={ckpt_smoke} python src/exp_nocaps.py --smoke --models climp-ours "
                        "--out-dir results_trained_smoke", "0")]]
elif STEP == "prep":
    STAGES = prep("")
elif STEP == "train":
    STAGES = [[("train", f"{TRAIN} --epochs {EPOCHS} --batch-size {BATCH} --max-hours {MAX_HOURS}"
                         + " --grad-ckpt" * GRAD_CKPT + " --resume" * RESUME, "0,1")]]
elif STEP == "eval":
    assert 1 <= len(EVAL) <= 2 and set(EVAL) <= set(EVALS), EVAL
    STAGES = [[(e, "CLIMP_OURS_CKPT={ckpt_full} " + EVALS[e], str(g)) for g, e in enumerate(EVAL)]]

LOGS.mkdir(exist_ok=True)
status, lock = [], threading.Lock()
say = lambda msg: (lock.acquire(), print(msg, flush=True), lock.release())

def last_line(path):
    lines = [l for l in path.read_text(errors="replace").replace("\r", "\n").splitlines() if l.strip()]
    return lines[-1][:200] if lines else ""

def job(name, cmd, gpus):
    if "{ckpt_" in cmd:
        cmd = cmd.format(ckpt_smoke=final_weights("smoke") if "ckpt_smoke" in cmd else "",
                         ckpt_full=final_weights("full") if "ckpt_full" in cmd else "")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus)
    say(f"[start] {name} (GPU {gpus or '-'}): {cmd}")
    t = time.time()
    with open(LOGS / f"{name}.log", "w") as f:
        code = subprocess.run(cmd, shell=True, cwd=REPO, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
    status.append((name, "ok" if code == 0 else f"FAILED ({code})", time.time() - t))
    tail = "\n".join((LOGS / f"{name}.log").read_text(errors="replace").replace("\r", "\n").splitlines()[-12:])
    say(f"[{'ok' if code == 0 else 'FAILED'}] {name} in {(time.time() - t) / 60:.1f} min. Last lines:\n{tail}\n")
    return code

start = time.time()
for stage in STAGES:
    threads = [threading.Thread(target=job, args=j) for j in stage]
    [t.start() for t in threads]
    last = time.time()
    while any(t.is_alive() for t in threads):
        if time.time() - last > 300:  # heartbeat: memory + newest log line of each running job
            gpu = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader", shell=True,
                                 capture_output=True, text=True).stdout.split()
            say(f"[{(time.time() - start) / 60:.0f} min] RAM {psutil.virtual_memory().used / 1e9:.1f} GB | "
                f"GPU MiB {' / '.join(gpu[0::2])}")
            for name, _, _ in stage:
                if (LOGS / f"{name}.log").exists():
                    say(f"    {name}: {last_line(LOGS / f'{name}.log')}")
            last = time.time()
        time.sleep(10)
    if any(s[1] != "ok" for s in status):
        say("a job failed -> later stages skipped")
        break
print(f"\nfinished in {(time.time() - start) / 60:.1f} min")
for name, st, secs in status:
    print(f"  {name:20s} {st:15s} {secs / 60:6.1f} min")

In [ ]:
name = f"train_{STEP}" + ("_" + "_".join(EVAL) if STEP == "eval" else "")
!rm -f /kaggle/working/{name}.zip && mkdir -p {REPO}/{OUT} && cp -r {LOGS} {REPO}/{OUT}/logs_{name} && cd {REPO} && zip -qr /kaggle/working/{name}.zip {OUT} -x "*/emb/*" && ls -lh /kaggle/working/{name}.zip